# Libraries

In [1]:
!pip -q install "lm_eval[hf]"
!pip -q install compressed-tensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 78.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.0 MB/s eta 0:00:00


In [2]:
# 2. Libraries
import random
import numpy as np
import torch
import os
import gc

import lm_eval
import subprocess

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import shutil
from IPython.display import FileLink

# Functions

In [3]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Global Variables

In [4]:
model_name = "EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit"
subfolder = "Sparsity/70"
SEED = 42
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Seed for reproducibility

In [5]:
set_reproducibility(SEED)

Global seed set to 42


# Login to huggingface

In [6]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Do Evaluation

**Configurations**

In [7]:
tasks_config = {
    "gsm8k": {
        "task_name": "gsm8k_cot",
        "fewshot": 2   # 🔻 reduced from 8
    },
    "mmlu": {
        "task_name": "mmlu",
        "fewshot": 2   # 🔻 reduced from 5
    },
    "arc_challenge": {
        "task_name": "arc_challenge",
        "fewshot": 0   # keep
    },
    "wikitext": {
        "task_name": "wikitext",
        "fewshot": 0   # keep
    }
}

In [8]:
for task, cfg in tasks_config.items():
    print(f"Starting evaluation for: {task}")

    model_args = f"pretrained={model_name},subfolder={subfolder},max_length=2048,dtype=float16,parallelize=True"
    command = [
        "lm_eval",
        "--model", "hf",
        "--model_args", model_args,
        "--tasks", cfg["task_name"],
        "--batch_size", str(3),   
        "--verbosity", "WARNING",
        "--seed", str(SEED),
        "--limit", str(100),
        "--num_fewshot", str(cfg["fewshot"]),
        "--output_path", f"evaluation_results/{task}"
    ]

    subprocess.run(command, check=True)
    
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Finished {task}\n")

Starting evaluation for: gsm8k


2026-03-28:14:05:38 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-28:14:05:44 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-03-28:14:05:44 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit appears to be an instruct or chat variant but chat template is not
        applied. Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-03-28 14:06:01.886338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774706762.250647     132 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774706762.353666     132 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register facto

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit', 'subfolder': 'Sparsity/70', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 3
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value|   |Stderr|
|---------|------:|----------------|-----:|-----------|---|----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     2|exact_match|↑  |    0|±  |     0|
|         |       |strict-match    |     2|exact_match|↑  |    0|±  |     0|

Finished gsm8k

Starting evaluation for: mmlu


2026-03-28:15:08:51 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-28:15:08:56 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-03-28:15:08:56 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit appears to be an instruct or chat variant but chat template is not
        applied. Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-03-28 15:09:03.125626: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774710543.147213     249 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774710543.152937     249 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory fo

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit', 'subfolder': 'Sparsity/70', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 3
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.2319|±  |0.0056|
| - humanities                          |      2|none  |      |acc   |↑  |0.2485|±  |0.0119|
|  - formal_logic                       |      1|none  |     2|acc   |↑  |0.2500|±  |0.0435|
|  - high_school_european_history       |      1|none  |     2|acc   |↑  |0.2500|±  |0.0435|
|  - high_school_us_history             |      1|none  |     2|acc   |↑  |0.2600|±  |0.0441|
|  - high_school_world_history          |      1|none  |     2|acc   |↑  |0.3000|±  |0.0461|
|  - international_law   

2026-03-28:15:37:55 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-28:15:38:00 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
2026-03-28:15:38:00 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit appears to be an instruct or chat variant but chat template is not
        applied. Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-03-28 15:38:07.546752: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774712287.567698    1001 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774712287.572997    1001 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register f

Finished arc_challenge

Starting evaluation for: wikitext


2026-03-28:15:39:21 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-28:15:39:26 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
2026-03-28:15:39:26 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT-AWQ-4bit appears to be an instruct or chat variant but chat template is not
        applied. Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
2026-03-28 15:39:33.908184: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774712373.928054    1077 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774712373.933112    1077 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factor

Finished wikitext



# Save reports

In [9]:
zip_path = shutil.make_archive(f"evaluation_results", 'zip', f"evaluation_results")
FileLink(zip_path)

/kaggle/working/evaluation_results.zip